In [1]:
import warnings
import sys

from pathlib import Path
from tqdm.auto import tqdm

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


PROJECT_ROOT = Path("/home/frantsuaza/Documents/Science/SportLab/collusion_judges")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import collusion_judges.config as config
from collusion_judges.config import  NODES_4_PATH, NODES_5_PATH, CORR_TYPES

from collusion_judges.modeling.analytics import make_group_comparison, make_ranked_judges_long, make_html_df_link


%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")


2026-10-29 23:26:53.615 | INFO     | collusion_judges.config:<module>:12 - PROJ_ROOT path is: /home/frantsuaza/Documents/Science/SportLab/collusion_judges


# I. Импорт датасета

Данный датасет содержит все данные по каждому судье. Всего тут 217 строк по количеству судей. Его может быть несколько трудновато читать из-за количества колонок. Далее я приведу датасет в более удобный для анализа вид. Тем не менее поясню как устроены колонки. Название колонки полностью отражает то, как была посчитана эта характеристика

```Python
CORR_TYPES = ("pearson", "kendall", "spearman")
SCORE_TYPES = ("components", "technique")
MICRO_STATS = ['mean', 'mad', 'rank_by_mean']
MACRO_STATS = ['median']

col_name = f'{macro_stata}_of_{score_type}_{corr_type}_{micro_stata}'
```

Лейбл `micro_stata` показывает, как агрегировалась информация на уровне 1 соревнования для этого судьи. Если написано `mean`, это значит, что посчиталась средняя корреляция этого судьи со всеми. Если написано `mad`, то посчитали разброс оценок для этого судьи в рамках 1 соревнования. `rank_by_mean` означает, что для каждого судьи посчитали среднюю корреляцию и проранжировали, сохранив ранг этого судьи (от 1 до 9) в рейтинге скоррелированности по этому событию.

Лейбл `macro_stata` показывает, как агрегировалась информация по 1 судье, который судил на нескольких соревнованиях. Для всех метрик просто бралась медиана. То есть `median_of_components_pearson_mad` означает, что для каждого соревнования, в котором участвовал судья, был посчитан разброс корреляции Пирсона `pearson_mad` по оценкам за компоненты, а потом от `components_pearson_mad` была взята медиана.

In [2]:
anomaly_df = pd.read_parquet(NODES_4_PATH)
anomaly_df

,judge_name,country,n_categories,n_events,median_of_components_pearson_fisher_mean,median_of_components_pearson_mad,median_of_components_pearson_rank_by_fisher_mean,median_of_technique_pearson_fisher_mean,median_of_technique_pearson_mad,median_of_technique_pearson_rank_by_fisher_mean,...,median_of_technique_kendall_rank_by_mean,anomaly_score_pearson_comp_tech,anomaly_rank_pearson_comp_tech,anomaly_score_spearman_comp_tech,anomaly_rank_spearman_comp_tech,anomaly_score_kendall_comp_tech,anomaly_rank_kendall_comp_tech,top10_anom_pearson,top10_anom_kendall,top10_anom_spearman
0,Lorrie PARKER,USA,3,4,0.867212,0.032278,4.5,0.923424,0.019558,7.5,...,7.0,0.399171,150,0.394967,168,0.391791,177,False,False,False
1,Susanne SJÖQUIST,SWE,2,2,0.929407,0.018486,4.0,0.945421,0.021663,7.0,...,8.0,0.421232,96,0.477946,42,0.481307,39,False,False,False
2,Steven HSU,USA,1,1,0.914509,0.026135,8.5,0.890667,0.025991,9.0,...,6.0,0.520993,16,0.608756,4,0.563150,8,False,True,True
3,Christian BAUMANN,DEU,1,1,0.916107,0.023733,6.5,0.669498,0.059300,7.5,...,6.5,0.553387,9,0.533236,16,0.540395,15,True,False,False
4,Etsuko AZUMA,JPN,3,3,0.838778,0.023617,5.5,0.948538,0.012307,4.5,...,3.5,0.423134,90,0.417790,111,0.412681,134,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,Lorna SCHRODER,CAN,1,1,0.884679,0.061586,2.5,0.936885,0.031392,4.5,...,5.0,0.460143,46,0.398478,155,0.428818,100,False,False,False
213,Luise Tarja BOEHLKE,DEU,1,1,0.908030,0.026597,6.5,0.856419,0.058814,5.5,...,6.0,0.435624,73,0.463795,59,0.532179,18,False,False,False
214,Jennifer MAST,USA,1,1,0.941239,0.017714,6.5,0.888550,0.030783,7.0,...,6.0,0.416465,101,0.429316,81,0.406958,143,False,False,False
215,Nicholas RUSSELL,GBR,1,1,0.882977,0.018484,6.0,0.766807,0.086077,8.0,...,7.5,0.551661,10,0.559241,12,0.513275,20,True,False,False


In [15]:
make_html_df_link(
    anomaly_df,
    filename="anomrating_wide.csv",
    label='Скачать датасет anomrating_wide.csv'
)

# II. Сравнение аномалий и неаномалий

В следующем агрегированном датасете можно посмотреть на средние значения в группе `топ-5` аномалий, `топ-10` аномалий и `other`, куда входят все за пределами топ-10. 

In [3]:
summary_df = make_group_comparison(anomaly_df)
display(summary_df.round(3))

data_description                         mean                       median  \
              Group corr_type  score_type   mean    mad rank_by_mean n_events   
0             top-5   pearson  components  0.722  0.118        6.000      1.0   
1             top-5   pearson   technique  0.754  0.086        5.300      1.0   
2            top-10   pearson  components  0.818  0.071        5.500      1.0   
3            top-10   pearson   technique  0.774  0.070        5.200      1.0   
4            others   pearson  components  0.884  0.034        5.092      3.0   
5            others   pearson   technique  0.906  0.026        5.126      3.0   
6             top-5   kendall  components  0.529  0.137        6.500      1.0   
7             top-5   kendall   technique  0.608  0.110        5.500      1.0   
8            top-10   kendall  components  0.585  0.104        5.850      1.0   
9            top-10   kendall   technique  0.645  0.085        4.950      1.0   
10           others   kendall  components  0.712  0.062        4.937      3.0   
11           others   kendall   technique  0.726  0.058        4.855      3.0   
12            top-5  spearman  components  0.645  0.128        6.000      1.0   
13            top-5  spearman   technique  0.710  0.095        5.400      1.0   
14           top-10  spearman  components  0.713  0.104        5.600      1.0   
15           top-10  spearman   technique  0.756  0.077        5.150      1.0   
16           others  spearman  components  0.834  0.046        4.986      3.0   
17           others  spearman   technique  0.855  0.040        4.959      3.0   

                 
   n_categories  
0           1.0  
1           1.0  
2           1.0  
3           1.0  
4           2.0  
5           2.0  
6           1.0  
7           1.0  
8           1.0  
9           1.0  
10          2.0  
11          2.0  
12          1.0  
13          1.0  
14          1.0  
15          1.0  
16          2.0  
17          2.0

In [16]:
make_html_df_link(
    summary_df,
    filename="comparison_summary.csv",
    label='Скачать датасет comparison_summary.csv'
)

# III. Детальный обзор аномальных судей

Вы можете использовать функцию `make_ranked_judges_long`, чтобы посмотреть на топ `n_judges` аномальных судей по какому-либо `corr_type`. Ниже будет датасет как в формате ниже, но он будет включать всех судей при всех корреляциях.

```Python
top5_kendall = make_ranked_judges_long(
    anomaly_df,
    corr_type="kendall",
    n_judges=5,
    save_path=None,
)
```

In [6]:
top5_kendall = make_ranked_judges_long(
    anomaly_df,
    corr_type="kendall",
    n_judges=5,
    save_path=None,
)
top5_kendall

score_descr                              judge                   \
    corr_type  score_type                   name country n_events   
0     kendall  components           Lucy LEONARD     GBR        1   
1     kendall   technique           Lucy LEONARD     GBR        1   
2     kendall  components             Hal MARRON     USA        1   
3     kendall   technique             Hal MARRON     USA        1   
4     kendall  components              Lynne DEY     CAN        1   
5     kendall   technique              Lynne DEY     CAN        1   
6     kendall  components         Tarja RISTANEN     FIN        2   
7     kendall   technique         Tarja RISTANEN     FIN        2   
8     kendall  components  Virpi KUNNAS-HELMINEN     FIN        5   
9     kendall   technique  Virpi KUNNAS-HELMINEN     FIN        5   

                                    median_of                         
  n_categories anom_rank anom_score      mean       mad rank_by_mean  
0            1         1   0.730343  0.559958  0.174938          5.0  
1            1         1   0.730343  0.476462  0.201170          4.5  
2            1         2   0.667972  0.321545  0.164190          7.0  
3            1         2   0.667972  0.741667  0.066667          7.5  
4            1         3   0.626026  0.556984  0.123518          7.0  
5            1         3   0.626026  0.455357  0.107143          5.0  
6            2         4   0.618351  0.492596  0.168719          5.0  
7            2         4   0.618351  0.833333  0.066667          2.5  
8            1         5   0.587011  0.713680  0.053782          8.5  
9            1         5   0.587011  0.534691  0.107149          8.0

In [7]:
long_df = pd.concat(
    [
        make_ranked_judges_long(
            anomaly_df,
            corr_type=corr_type,
            n_judges=None,
            save_path=None,
        )
        for corr_type in CORR_TYPES
    ],
    ignore_index=True,
)

display(long_df)

score_descr                         judge                                \
       corr_type  score_type              name country n_events n_categories   
0        pearson  components      Lucy LEONARD     GBR        1            1   
1        pearson   technique      Lucy LEONARD     GBR        1            1   
2        pearson  components        Hal MARRON     USA        1            1   
3        pearson   technique        Hal MARRON     USA        1            1   
4        pearson  components         Lynne DEY     CAN        1            1   
...          ...         ...               ...     ...      ...          ...   
1297    spearman   technique  Claudia BRAMBATI     ITA        4            2   
1298    spearman  components  Ritsuko HORIUCHI     JPN        4            3   
1299    spearman   technique  Ritsuko HORIUCHI     JPN        4            3   
1300    spearman  components   Kyungwon SAKONG     KOR        6            3   
1301    spearman   technique   Kyungwon SAKONG     KOR        6            3   

                          median_of                         
     anom_rank anom_score      mean       mad rank_by_mean  
0            1   0.754530  0.754037  0.152761          5.5  
1            1   0.754530  0.600856  0.187723          6.5  
2            2   0.716480  0.500244  0.222747          7.0  
3            2   0.716480  0.854818  0.033430          5.5  
4            3   0.614240  0.788613  0.055762          6.0  
...        ...        ...       ...       ...          ...  
1297       215   0.367759  0.872768  0.035714          5.0  
1298       216   0.366396  0.805096  0.044971          5.5  
1299       216   0.366396  0.876711  0.029895          5.5  
1300       217   0.363345  0.820933  0.044749          5.5  
1301       217   0.363345  0.863728  0.033217          5.5  

[1302 rows x 11 columns]

In [14]:
make_html_df_link(
    long_df,
    filename="anomrating_long.csv",
    label='Скачать датасет anomrating_long.csv'
)